# Pre subida al Postgres

### Archivos de entrada
| Archivo | Contenido |
|---|---|
| `esea_meta_demos.part1.csv` | Metadata de partidas y rondas |
| `esea_master_dmg_demos.part1.csv` | Eventos de daño por tick |
| `esea_master_kills_demos.part1.csv` | Eventos de kill por tick |

### Tablas
**Entidades:** `Partida`, `Ronda`, `Arma`, `Jugador`, `Ticks`  
**Relaciones:** `Participacion`, `Danno` (danno en vez de daño pq se puede chingar el postgres)

In [2]:
import pandas as pd
dmeta  = pd.read_csv("esea_meta_demos.part1.csv")
ddmg   = pd.read_csv("esea_master_dmg_demos.part1.csv")
dkills = pd.read_csv("esea_master_kills_demos.part1.csv")

---
## Entidades
### 1. Partida

In [128]:
Partida = (
    dmeta[["file", "map"]]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy().rename(columns={
        "file":  "Archivo",
        "map" : "Mapa"
    })
)

Partida.head()

,Archivo,Mapa
0,esea_match_13770997.dem,de_overpass
1,esea_match_13779704.dem,de_cache
2,esea_match_13779769.dem,de_inferno
3,esea_match_13779770.dem,de_mirage
4,esea_match_13779771.dem,de_inferno


### 2. Ronda
Una fila por ronda. `esea_meta_demos` ya tiene una fila por ronda, solo se renombran columnas.

Se incluyen `ct_eq_val` y `t_eq_val` (economía de cada equipo al inicio de la ronda) porque son parte del análisis principal del proyecto.

In [115]:
Ronda = (
    dmeta[["file", "round", "winner_team", "winner_side",
           "ct_eq_val", "t_eq_val", "round_type"]]
    .copy()
    .rename(columns={
        "file":  "Partida",
        "round": "Número",
        "round_type" : "Tipo_de_ronda",
        "ct_eq_val" : "CT_eq_val",
        "t_eq_val" : "T_eq_val"
    })
)

Ronda.head()

,Partida,Número,winner_team,winner_side,CT_eq_val,T_eq_val,Tipo_de_ronda
0,esea_match_13770997.dem,1,Hentai Hooligans,Terrorist,4300,4250,PISTOL_ROUND
1,esea_match_13770997.dem,2,Hentai Hooligans,Terrorist,6300,19400,ECO
2,esea_match_13770997.dem,3,Hentai Hooligans,Terrorist,7650,19250,SEMI_ECO
3,esea_match_13770997.dem,4,Hentai Hooligans,Terrorist,24900,23400,NORMAL
4,esea_match_13770997.dem,5,Animal Style,CounterTerrorist,5400,20550,ECO


### 3. Arma


In [132]:
Arma = (
    ddmg[["wp", "wp_type"]]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
    .rename(columns={
        "wp":      "Nombre",
        "wp_type": "Tipo"
    })
)

Arma = Arma[Arma['Nombre'] != 'Unknown']

Arma.head()

,Nombre,Tipo
1,USP,Pistol
2,Glock,Pistol
3,HE,Grenade
4,Deagle,Pistol
5,AK47,Rifle


### 4. Jugador
Los jugadores aparecen como `att_id` y `vic_id` en cada fila de daño.

Se unen ambas columnas, se eliminan duplicados y se filtra `id = 0` que representa al daño por caída, fuego, etc.

In [135]:
Jugador = (
    pd.concat([
        ddmg[["att_id"]].rename(columns={"att_id": "id"}),
        ddmg[["vic_id"]].rename(columns={"vic_id": "id"})
    ], ignore_index=True)
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
    .rename(columns={
        "id" : "ID"
    })
)

Jugador = Jugador[Jugador["ID"] != 0].copy()
Jugador["ID"] = Jugador["ID"].astype("int64")

Jugador.head()

,ID
1,76561198048742997
2,76561198055054795
3,76561198082200410
4,76561198055191021
5,76561198037331400


### 5. Ticks
Se reconstruye cruzando:
- `ddmg` → estado de la bomba (`is_bomb_planted`, `bomb_site`) por tick
- `dkills` → jugadores vivos (`ct_alive`, `t_alive`) por tick

 `outer join` para no perder ticks que aparezcan solo en una de las dos fuentes.

In [ ]:
# Estado de bomba: un tick puede tener múltiples eventos de daño
# pero el estado de la bomba es el mismo en todos → drop_duplicates antes del merge
ticks_bomba = (
    ddmg[["file", "round", "tick", "seconds", "is_bomb_planted", "bomb_site"]]
    .drop_duplicates(subset=["file", "round", "tick"])
)

# Jugadores vivos: solo cambia en ticks donde hubo un kill
ticks_vivos = (
    dkills[["file", "round", "tick", "ct_alive", "t_alive"]]
    .drop_duplicates(subset=["file", "round", "tick"])
)

Ticks = (
    pd.merge(ticks_bomba, ticks_vivos, on=["file", "round", "tick"], how="outer")
    .rename(columns={
        "tick":  "ID",
        "seconds": "Segundo",
        "file" : 'Partida',
        'is_bomb_planted' : 'Bomb_planted',
        'bomb_site' : 'Bomb_site',
        'ct_alive' : 'CT_alive',
        't_alive' : 'T_alive'
    })
)

Ticks = Ticks[['ID', 'Segundo', 'Partida', 'Bomb_planted', 'Bomb_site', 'CT_alive', 'T_alive']]


Ticks.head()

,ID,Segundo,Partida,Bomb_planted,Bomb_site,CT_alive,T_alive
0,14372,111.8476,esea_match_13770997.dem,False,NaN,NaN,NaN
1,15972,124.3761,esea_match_13770997.dem,False,NaN,NaN,NaN
2,16058,125.0495,esea_match_13770997.dem,False,NaN,5.0,4.0
3,16066,125.1121,esea_match_13770997.dem,False,NaN,NaN,NaN
4,16108,125.4410,esea_match_13770997.dem,False,NaN,NaN,NaN


In [ ]:
## Revisando que tick no es entidad debil de ronda

temp = Ticks[Ticks['archivo'] == 'esea_match_13770997.dem']

temp = temp.groupby(['numero_ronda']).min(['tiempo'])

temp

,numero,segundo,ct_alive,t_alive
numero_ronda,,,,
1,14372,111.8476,0.0,1.0
2,23922,186.5484,1.0,2.0
3,40775,318.2537,0.0,4.0
4,46840,365.5798,0.0,1.0
5,58883,459.7310,3.0,0.0
6,65643,512.6011,0.0,1.0
7,82292,642.7402,0.0,3.0
8,91571,715.2642,1.0,0.0
9,102203,798.3591,3.0,0.0


---
## Relaciones
### 6. Participacion
Qué jugador participó en qué partida y para qué equipo.

Un jugador cambia de lado exactamente en la ronda 16 (CT ↔ T), por lo que aparece en ambos lados dentro de la misma partida. Se usa `mode()` para quedarse con el equipo más frecuente, que corresponde al equipo principal del jugador en esa partida.

In [106]:
Participacion = (
    ddmg[["file", "att_id", "att_team"]]
    [ddmg["att_id"] != 0]
    .groupby(["file", "att_id"])["att_team"]
    .agg(lambda x: x.min())
    .reset_index()
    .rename(columns={
        "file":     "Archivo",
        "att_id":   "ID_jugador",
        "att_team": "Equipo"
    })
)

Participacion.head()

,Archivo,ID_jugador,Equipo
0,esea_match_13770997.dem,76561197961009213,Hentai Hooligans
1,esea_match_13770997.dem,76561198037331400,Hentai Hooligans
2,esea_match_13770997.dem,76561198048742997,Animal Style
3,esea_match_13770997.dem,76561198055054795,Animal Style
4,esea_match_13770997.dem,76561198055191021,Animal Style


### 7. Danno
Cada evento de daño entre dos jugadores en un tick específico.

La columna `mata` sale cruzando con `dkills`: si existe un kill en el mismo `(file, round, tick)`, el evento de daño resultó en muerte.

In [140]:
## Merge kills con daño para obtener atributo Es_fatal

Kills = dkills[['file', 'round', 'tick']].assign(Es_fatal=True)

Danno = (
    pd.merge(ddmg, Kills, on=["file", "round", "tick"], how="left")
)
Danno["Es_fatal"] = Danno["Es_fatal"].infer_objects(copy=False).fillna(False)

Danno = Danno[['wp','tick', 'att_id', 'vic_id', 'Es_fatal', 'hitbox', 'hp_dmg', 'arm_dmg', 'att_pos_x','vic_pos_y', 'vic_pos_x', 'vic_pos_y', 'att_side', 'vic_side']]


Danno = Danno.rename(columns={
    "hitbox" : "Hitbox",
    "att_id" : "Atacante",
    "vic_id": "Victima",
    "wp":      "Arma",
    "hp_dmg":  "HP_dmg",
    "arm_dmg": "Armor_dmg",
    "att_side": "Lado_att",
    "vic_side": "Lado_vic",
    "att_pos_x": "Att_pos_x",
    "att_pos_y": "Att_pos_y",
    "vic_pos_x": "Vic_pos_x",
    "vic_pos_y": "Vic_pos_y"
})

Danno = Danno[Danno['Arma'] != 'Unknown'] ## Siempre debemos saber el arma
Danno = Danno[(Danno['Atacante']!= 0)&(Danno['Victima']!= 0)] ## Siempre sabemos los jugadores


Danno.head()

C:\Users\HP\AppData\Local\Temp\ipykernel_26256\1110501942.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Danno["Es_fatal"] = Danno["Es_fatal"].infer_objects(copy=False).fillna(False)


,Arma,tick,Atacante,Victima,Es_fatal,Hitbox,HP_dmg,Armor_dmg,Att_pos_x,Vic_pos_y,Vic_pos_x,Vic_pos_y,Lado_att,Lado_vic
1,USP,15972,76561198048742997,76561198082200410,False,Stomach,18,9,-1499.6900,-79.769570,-669.5558,-79.769570,CounterTerrorist,Terrorist
2,USP,16058,76561198055054795,76561197961009213,True,Head,100,0,-1066.8740,-91.707770,-614.1868,-91.707770,CounterTerrorist,Terrorist
3,Glock,16066,76561198082200410,76561198055054795,False,RightArm,12,7,-747.3146,9.381622,-1065.5560,9.381622,Terrorist,CounterTerrorist
4,USP,16108,76561198048742997,76561198082200410,False,Chest,15,7,-1501.8610,-53.469220,-748.4188,-53.469220,CounterTerrorist,Terrorist
5,Glock,16188,76561198082200410,76561198048742997,False,Head,94,0,-756.2186,60.658150,-1500.0780,60.658150,Terrorist,CounterTerrorist


## Transcurre en

In [ ]:
### Tabla para hacer el cruce de tick con Ronda

Transcurre_en = ddmg[['file', 'tick', 'round']]

Transcurre_en.head()


,file,tick,round
0,esea_match_13770997.dem,14372,1
1,esea_match_13770997.dem,15972,1
2,esea_match_13770997.dem,16058,1
3,esea_match_13770997.dem,16066,1
4,esea_match_13770997.dem,16108,1


---
## Exportar CSVs

In [144]:
import os
os.makedirs("output", exist_ok=True)

tablas = {
    "partida":       Partida,
    "ronda":         Ronda,
    "arma":          Arma,
    "jugador":       Jugador,
    "ticks":         Ticks,
    "participacion": Participacion,
    "danno":         Danno,
    "Transcurre_en": Transcurre_en
}

for nombre, df in tablas.items():
    ruta = f"output/{nombre}.csv"
    df.to_csv(ruta, index=False)
    print(f"{nombre:15s} → {ruta}  ({len(df):,} filas)")

partida         → output/partida.csv  (8,527 filas)
ronda           → output/ronda.csv  (215,919 filas)
arma            → output/arma.csv  (41 filas)
jugador         → output/jugador.csv  (17,878 filas)
ticks           → output/ticks.csv  (5,778,260 filas)
participacion   → output/participacion.csv  (86,869 filas)
danno           → output/danno.csv  (5,938,244 filas)
Transcurre_en   → output/Transcurre_en.csv  (5,992,097 filas)
